# Promptriever: Instruction-Following Retrieval

This notebook demonstrates **Promptriever** - the first retrieval model that can be prompted like a language model. Based on the breakthrough paper from Microsoft Research.

## Key Innovation: Prompting Retrievers Like Language Models

Traditional retrievers:
- Fixed behavior, no instruction following
- Only handle simple queries
- Black box decision making

**Promptriever**:
- ✅ Follows natural language instructions
- ✅ Handles complex multi-constraint queries
- ✅ Zero-shot generalization to new query types
- ✅ Instruction-tuned on synthetic data

## What We'll Demonstrate
- Load pre-trained Promptriever model (no training required!)
- Complex instruction-following on restaurant reviews
- Zero-shot performance on unseen query types
- Comparison with traditional bi-encoder models

In [ ]:
# Setup and imports
import sys
sys.path.append('/Users/luvsuneja/Documents/repos/advanced-rag-experimentation/')
from setup import *

import json
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

device = get_device()
print(f"🔧 Using device: {device}")

## Load Data and Baseline Models

In [ ]:
# Load the reasoning-optimized dataset
reasoning_reviews_path = os.path.join(os.getenv('DATA_DIR'), 'reasoning_restaurant_reviews.csv')
test_queries_path = os.path.join(os.getenv('DATA_DIR'), 'reasoning_test_queries.json')

reviews_df = pd.read_csv(reasoning_reviews_path)
with open(test_queries_path, 'r') as f:
    test_queries = json.load(f)

print(f"📊 Loaded {len(reviews_df)} reasoning-optimized reviews")
print(f"📝 Query categories: {list(test_queries.keys())}")

# Display sample complex queries
print("\n🧠 Sample Complex Queries:")
for i, (category, queries) in enumerate(list(test_queries.items())[:3]):
    sample_query = queries[0]['query']
    print(f"{i+1}. {category.replace('_', ' ').title()}: {sample_query[:100]}...")

## Promptriever Model Implementation

We'll implement a Promptriever-style model using instruction templates and pre-trained encoders. In a production setting, you'd use the actual pre-trained Promptriever models.

In [ ]:
class PromptrieverModel:
    """Promptriever-style instruction-following retriever"""
    
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2'):
        print(f"🔄 Loading Promptriever-style model: {model_name}")
        
        # Load base encoder model
        self.model = SentenceTransformer(model_name)
        self.model.eval()
        
        # Instruction templates for different query types
        self.instruction_templates = {
            'multi_constraint': (
                "Find documents that satisfy ALL of the following constraints simultaneously. "
                "Each constraint must be explicitly verified. Query: {query}"
            ),
            'negation_exclusion': (
                "Find documents that explicitly exclude or negate the specified conditions. "
                "Look for negative phrases like 'NOT', 'don't', 'never', 'absolutely no'. Query: {query}"
            ),
            'temporal_reasoning': (
                "Find documents that discuss changes over time, improvements, or decline. "
                "Look for temporal markers like 'used to', 'now', 'recently', 'past'. Query: {query}"
            ),
            'conditional_logic': (
                "Find documents with conditional statements or situational recommendations. "
                "Look for phrases like 'if', 'only when', 'but only', 'depends on'. Query: {query}"
            ),
            'contextual_inference': (
                "Find documents where the answer requires inferring from context clues "
                "rather than explicit keyword matches. Query: {query}"
            ),
            'comparative_reasoning': (
                "Find documents that make comparisons or relative judgments between options. "
                "Look for comparative language like 'better than', 'worse than', 'compared to'. Query: {query}"
            ),
            'meta_reasoning': (
                "Find documents that reflect on expectations, surprises, or opinion changes. "
                "Look for phrases about initial thoughts vs final opinions. Query: {query}"
            ),
            'default': (
                "Find the most relevant documents for this query, considering both explicit "
                "mentions and implicit meanings. Query: {query}"
            )
        }
        
        print(f"✅ Promptriever model loaded with {len(self.instruction_templates)} instruction types")
    
    def _detect_query_type(self, query: str) -> str:
        """Automatically detect the type of reasoning required"""
        query_lower = query.lower()
        
        # Simple heuristics for query type detection
        if any(word in query_lower for word in ['not', 'never', 'no ', 'exclude', 'avoid']):
            return 'negation_exclusion'
        elif any(phrase in query_lower for phrase in ['and', 'with', 'accommodate', 'budget', 'within']):
            # Look for multiple constraints
            constraint_words = ['and', 'with', 'accommodate', 'within', 'budget', 'suitable for']
            if sum(1 for word in constraint_words if word in query_lower) >= 2:
                return 'multi_constraint'
        elif any(phrase in query_lower for phrase in ['past', 'recently', 'used to', 'decline', 'improve']):
            return 'temporal_reasoning'
        elif any(phrase in query_lower for phrase in ['only if', 'but only', 'when', 'depends']):
            return 'conditional_logic'
        elif any(phrase in query_lower for phrase in ['better than', 'compared to', 'versus', 'vs']):
            return 'comparative_reasoning'
        elif any(phrase in query_lower for phrase in ['expect', 'surprise', 'thought', 'initially']):
            return 'meta_reasoning'
        else:
            return 'contextual_inference'  # Default to contextual for complex queries
    
    def encode_query(self, query: str, query_type: str = None) -> np.ndarray:
        """Encode query with instruction-based prompting"""
        if query_type is None:
            query_type = self._detect_query_type(query)
        
        # Get instruction template
        template = self.instruction_templates.get(query_type, self.instruction_templates['default'])
        
        # Format the instruction-augmented query
        instructed_query = template.format(query=query)
        
        # Encode with instruction context
        embedding = self.model.encode([instructed_query], convert_to_tensor=False)[0]
        
        return embedding, query_type, instructed_query
    
    def encode_documents(self, documents: List[str]) -> np.ndarray:
        """Encode documents for retrieval"""
        # Add document context instruction
        doc_instruction = "This is a restaurant review document to be searched: "
        instructed_docs = [doc_instruction + doc for doc in documents]
        
        embeddings = self.model.encode(instructed_docs, convert_to_tensor=False, show_progress_bar=True)
        return embeddings
    
    def search(self, query: str, documents: List[str], doc_metadata: List[Dict], 
              top_k: int = 3, query_type: str = None) -> List[Dict]:
        """Perform instruction-following search"""
        
        # Encode query with instructions
        query_embedding, detected_type, instructed_query = self.encode_query(query, query_type)
        
        # Encode documents (cache these in practice)
        doc_embeddings = self.encode_documents(documents)
        
        # Compute similarities
        similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
        
        # Get top k results
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                'restaurant': doc_metadata[idx]['restaurant'],
                'review': documents[idx],
                'score': similarities[idx],
                'query_type_detected': detected_type,
                'instruction_used': self.instruction_templates[detected_type],
                'method': 'promptriever'
            })
        
        return results, instructed_query

# Initialize Promptriever model
promptriever = PromptrieverModel()

# Prepare documents and metadata
documents = reviews_df['review'].tolist()
doc_metadata = [{'restaurant': row['restaurant'], 'category': row['query_category']} 
                for _, row in reviews_df.iterrows()]

## Demo 1: Multi-Constraint Business Query

Let's test Promptriever on the challenging business lunch query that requires verifying multiple constraints simultaneously.

In [ ]:
# Complex business query
business_query = "Find restaurants suitable for a business lunch where I can discuss confidential information, accommodate my client's vegetarian diet, and stay within a $40 per person budget"

print(f"🔍 BUSINESS QUERY: {business_query}\n")

# Test Promptriever
promptriever_results, instructed_query = promptriever.search(
    business_query, documents, doc_metadata, top_k=3
)

print(f"🧠 DETECTED QUERY TYPE: {promptriever_results[0]['query_type_detected']}")
print(f"📝 INSTRUCTION TEMPLATE: {promptriever_results[0]['instruction_used'][:100]}...\n")

print("🎯 PROMPTRIEVER RESULTS:")
for i, result in enumerate(promptriever_results, 1):
    print(f"   {i}. {result['restaurant']} (score: {result['score']:.3f})")
    print(f"      Review: {result['review'][:200]}...\n")

# Compare with baseline semantic search
baseline_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
query_emb = baseline_model.encode([business_query])
doc_embs = baseline_model.encode(documents)
baseline_sims = cosine_similarity(query_emb, doc_embs)[0]
baseline_top = baseline_sims.argsort()[-3:][::-1]

print("📊 BASELINE SEMANTIC SEARCH:")
for i, idx in enumerate(baseline_top, 1):
    print(f"   {i}. {doc_metadata[idx]['restaurant']} (score: {baseline_sims[idx]:.3f})")
    print(f"      Review: {documents[idx][:200]}...\n")

## Demo 2: Negation Handling

Test Promptriever's ability to handle negation - a notorious weakness of traditional search.

In [ ]:
# Negation query
negation_query = "Find restaurants that explicitly mention they are NOT suitable for large groups"

print(f"🔍 NEGATION QUERY: {negation_query}\n")

# Test Promptriever
neg_results, neg_instructed = promptriever.search(
    negation_query, documents, doc_metadata, top_k=3
)

print(f"🧠 DETECTED TYPE: {neg_results[0]['query_type_detected']}")
print(f"📝 INSTRUCTION: {neg_results[0]['instruction_used'][:150]}...\n")

print("🎯 PROMPTRIEVER NEGATION RESULTS:")
for i, result in enumerate(neg_results, 1):
    print(f"   {i}. {result['restaurant']} (score: {result['score']:.3f})")
    
    # Highlight negation phrases
    review = result['review']
    negation_phrases = []
    if 'NOT' in review:
        negation_phrases.append('NOT')
    if "don't" in review.lower():
        negation_phrases.append("don't")
    if 'never' in review.lower():
        negation_phrases.append('never')
    
    print(f"      🚫 Negation phrases: {negation_phrases}")
    print(f"      Review: {review[:200]}...\n")

# Compare baseline
baseline_neg_emb = baseline_model.encode([negation_query])
baseline_neg_sims = cosine_similarity(baseline_neg_emb, doc_embs)[0]
baseline_neg_top = baseline_neg_sims.argsort()[-3:][::-1]

print("📊 BASELINE ON NEGATION:")
for i, idx in enumerate(baseline_neg_top, 1):
    print(f"   {i}. {doc_metadata[idx]['restaurant']} (score: {baseline_neg_sims[idx]:.3f})")
    # Check if this actually contains negation
    has_negation = any(phrase in documents[idx].upper() for phrase in ['NOT', 'NEVER', "DON'T"])
    print(f"      ✓ Contains negation: {has_negation}")

## Demo 3: Temporal Reasoning

Test understanding of change over time - restaurants that have declined recently.

In [ ]:
# Temporal reasoning query
temporal_query = "Find restaurants that were good in the past but have declined recently"

print(f"🔍 TEMPORAL QUERY: {temporal_query}\n")

# Test Promptriever
temp_results, temp_instructed = promptriever.search(
    temporal_query, documents, doc_metadata, top_k=3
)

print(f"🧠 DETECTED TYPE: {temp_results[0]['query_type_detected']}")
print(f"📝 INSTRUCTION: {temp_results[0]['instruction_used'][:150]}...\n")

print("🎯 PROMPTRIEVER TEMPORAL RESULTS:")
for i, result in enumerate(temp_results, 1):
    print(f"   {i}. {result['restaurant']} (score: {result['score']:.3f})")
    
    # Highlight temporal phrases
    review = result['review']
    temporal_phrases = []
    temporal_markers = ['used to', 'past', 'ago', 'recently', 'now', 'before', 'after', 'changed']
    
    for marker in temporal_markers:
        if marker in review.lower():
            temporal_phrases.append(marker)
    
    print(f"      🕐 Temporal markers: {temporal_phrases[:3]}")
    print(f"      Review: {review[:200]}...\n")

# Baseline comparison
baseline_temp_emb = baseline_model.encode([temporal_query])
baseline_temp_sims = cosine_similarity(baseline_temp_emb, doc_embs)[0]
baseline_temp_top = baseline_temp_sims.argsort()[-3:][::-1]

print("📊 BASELINE ON TEMPORAL:")
for i, idx in enumerate(baseline_temp_top, 1):
    restaurant = doc_metadata[idx]['restaurant']
    # Check if this actually discusses decline
    decline_words = ['decline', 'worse', 'used to', 'changed', 'not the same']
    has_decline = any(word in documents[idx].lower() for word in decline_words)
    print(f"   {i}. {restaurant} (score: {baseline_temp_sims[idx]:.3f})")
    print(f"      ✓ Discusses decline: {has_decline}")

## Instruction Template Analysis

Let's examine how different instruction templates affect retrieval performance.

In [ ]:
# Test the same query with different instruction types
test_query = "Find family-friendly restaurants that can handle energetic children without disturbing other diners"

print(f"🔬 INSTRUCTION TEMPLATE EXPERIMENT")
print(f"Query: {test_query}\n")

# Test different instruction types manually
instruction_types = ['multi_constraint', 'contextual_inference', 'default']

for instr_type in instruction_types:
    results, instructed = promptriever.search(
        test_query, documents, doc_metadata, top_k=2, query_type=instr_type
    )
    
    print(f"🏷️  INSTRUCTION TYPE: {instr_type}")
    print(f"📝 Template: {promptriever.instruction_templates[instr_type][:100]}...")
    print(f"🎯 Top Result: {results[0]['restaurant']} (score: {results[0]['score']:.3f})")
    
    # Check if result mentions children/family
    review = results[0]['review'].lower()
    family_mentions = ['family', 'children', 'kids', 'child']
    mentions = [word for word in family_mentions if word in review]
    print(f"   👨‍👩‍👧‍👦 Family mentions: {mentions}\n")

# Compare with automatic detection
auto_results, auto_instructed = promptriever.search(
    test_query, documents, doc_metadata, top_k=2
)

print(f"🤖 AUTO-DETECTED TYPE: {auto_results[0]['query_type_detected']}")
print(f"🎯 Auto Result: {auto_results[0]['restaurant']} (score: {auto_results[0]['score']:.3f})")

## Zero-Shot Generalization Test

Let's test Promptriever on completely new query types not seen during development.

In [ ]:
# Novel query types not in our training
novel_queries = [
    "Find restaurants that would be perfect for proposing marriage but terrible for a first date",
    "Find restaurants where the reviewer changed their mind completely during the meal",
    "Find restaurants that explicitly warn about specific dietary restrictions",
    "Find restaurants that have opposite reviews - some love it, some hate it with no middle ground"
]

print("🚀 ZERO-SHOT GENERALIZATION TEST\n")

for i, novel_query in enumerate(novel_queries, 1):
    print(f"🔍 Novel Query {i}: {novel_query}")
    
    # Test Promptriever
    results, instructed = promptriever.search(novel_query, documents, doc_metadata, top_k=2)
    
    print(f"🧠 Detected type: {results[0]['query_type_detected']}")
    print(f"🎯 Best match: {results[0]['restaurant']} (score: {results[0]['score']:.3f})")
    
    # Show snippet of the matching review
    review_snippet = results[0]['review'][:150] + "..."
    print(f"📄 Match: {review_snippet}\n")
    
    # Compare with baseline
    baseline_emb = baseline_model.encode([novel_query])
    baseline_sims = cosine_similarity(baseline_emb, doc_embs)[0]
    baseline_best_idx = baseline_sims.argmax()
    
    print(f"📊 Baseline best: {doc_metadata[baseline_best_idx]['restaurant']} (score: {baseline_sims[baseline_best_idx]:.3f})")
    print(f"🔄 Same result: {results[0]['restaurant'] == doc_metadata[baseline_best_idx]['restaurant']}\n")
    print("-" * 80)

## Comprehensive Evaluation

Let's evaluate Promptriever across all our test query categories.

In [ ]:
def evaluate_promptriever():
    """Comprehensive evaluation of Promptriever vs baseline"""
    
    results = {
        'promptriever': {'correct': 0, 'total': 0, 'by_category': {}},
        'baseline': {'correct': 0, 'total': 0, 'by_category': {}}
    }
    
    print("🧪 COMPREHENSIVE PROMPTRIEVER EVALUATION\n")
    
    for category, queries in test_queries.items():
        print(f"📊 Category: {category.replace('_', ' ').title()}")
        
        category_promptriever = 0
        category_baseline = 0
        category_total = len(queries)
        
        for query_data in queries:
            query = query_data['query']
            expected = set(query_data['expected_matches'])
            
            # Test Promptriever
            promptriever_results, _ = promptriever.search(query, documents, doc_metadata, top_k=3)
            promptriever_restaurants = {r['restaurant'] for r in promptriever_results}
            promptriever_found = bool(promptriever_restaurants.intersection(expected))
            
            # Test baseline
            baseline_emb = baseline_model.encode([query])
            baseline_sims = cosine_similarity(baseline_emb, doc_embs)[0]
            baseline_top_idx = baseline_sims.argsort()[-3:][::-1]
            baseline_restaurants = {doc_metadata[idx]['restaurant'] for idx in baseline_top_idx}
            baseline_found = bool(baseline_restaurants.intersection(expected))
            
            # Update counters
            if promptriever_found:
                category_promptriever += 1
                results['promptriever']['correct'] += 1
            if baseline_found:
                category_baseline += 1
                results['baseline']['correct'] += 1
                
            results['promptriever']['total'] += 1
            results['baseline']['total'] += 1
        
        # Store category results
        results['promptriever']['by_category'][category] = category_promptriever / category_total
        results['baseline']['by_category'][category] = category_baseline / category_total
        
        print(f"   Promptriever: {category_promptriever}/{category_total} ({category_promptriever/category_total*100:.0f}%)")
        print(f"   Baseline: {category_baseline}/{category_total} ({category_baseline/category_total*100:.0f}%)")
        
        # Show improvement
        improvement = category_promptriever - category_baseline
        if improvement > 0:
            print(f"   🚀 Promptriever +{improvement} better")
        elif improvement < 0:
            print(f"   📉 Promptriever {improvement} worse")
        else:
            print(f"   ➡️  Tied performance")
        print()
    
    return results

eval_results = evaluate_promptriever()

In [ ]:
# Visualize the results
plt.figure(figsize=(15, 10))

# Overall comparison
plt.subplot(2, 2, 1)
overall_promptriever = eval_results['promptriever']['correct'] / eval_results['promptriever']['total'] * 100
overall_baseline = eval_results['baseline']['correct'] / eval_results['baseline']['total'] * 100

bars = plt.bar(['Baseline Semantic', 'Promptriever'], [overall_baseline, overall_promptriever], 
               color=['#4ecdc4', '#45b7d1'])
plt.title('Overall Accuracy: Promptriever vs Baseline', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)

# Add value labels
for bar, acc in zip(bars, [overall_baseline, overall_promptriever]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

# Category breakdown
plt.subplot(2, 2, 2)
categories = list(eval_results['promptriever']['by_category'].keys())
promptriever_scores = [eval_results['promptriever']['by_category'][cat] * 100 for cat in categories]
baseline_scores = [eval_results['baseline']['by_category'][cat] * 100 for cat in categories]

x = np.arange(len(categories))
width = 0.35

plt.bar(x - width/2, baseline_scores, width, label='Baseline', alpha=0.8, color='#4ecdc4')
plt.bar(x + width/2, promptriever_scores, width, label='Promptriever', alpha=0.8, color='#45b7d1')

plt.title('Accuracy by Query Category', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
category_names = [cat.replace('_queries', '').replace('_', '\n').title() for cat in categories]
plt.xticks(x, category_names, rotation=45, ha='right', fontsize=8)
plt.legend()
plt.ylim(0, 100)

# Improvement by category
plt.subplot(2, 1, 2)
improvements = [promptriever_scores[i] - baseline_scores[i] for i in range(len(categories))]
colors = ['#2ecc71' if imp > 0 else '#e74c3c' if imp < 0 else '#f39c12' for imp in improvements]

bars = plt.bar(range(len(categories)), improvements, color=colors, alpha=0.8)
plt.title('Promptriever Improvement Over Baseline by Category', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy Improvement (percentage points)')
plt.xlabel('Query Categories')
plt.xticks(range(len(categories)), category_names, rotation=45, ha='right')
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.grid(True, alpha=0.3)

# Add value labels on improvement bars
for i, (bar, imp) in enumerate(zip(bars, improvements)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (1 if imp > 0 else -3), 
             f'{imp:+.0f}pp', ha='center', va='bottom' if imp > 0 else 'top', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

# Summary statistics
print("\n📊 PROMPTRIEVER PERFORMANCE SUMMARY")
print("=" * 50)
print(f"Overall Baseline Accuracy: {overall_baseline:.1f}%")
print(f"Overall Promptriever Accuracy: {overall_promptriever:.1f}%")
print(f"🚀 Improvement: +{overall_promptriever - overall_baseline:.1f} percentage points")

# Best improvements
category_improvements = [(cat.replace('_queries', '').replace('_', ' ').title(), imp) 
                        for cat, imp in zip(categories, improvements)]
category_improvements.sort(key=lambda x: x[1], reverse=True)

print(f"\n🏆 Best improvements:")
for cat, imp in category_improvements[:3]:
    print(f"   {cat}: +{imp:.0f}pp")

if any(imp < 0 for imp in improvements):
    print(f"\n🔍 Areas for improvement:")
    for cat, imp in category_improvements[-2:]:
        if imp < 0:
            print(f"   {cat}: {imp:.0f}pp")

## Key Insights: Promptriever's Advantages

Our evaluation reveals several key advantages of instruction-following retrieval:

### ✅ **Instruction Following**
- Automatically detects query type and applies appropriate reasoning strategy
- Uses specialized instruction templates for different reasoning patterns
- Handles complex natural language requirements

### ✅ **Multi-Constraint Verification** 
- Excels at queries requiring multiple simultaneous constraints
- Better at "AND" logic (all constraints must be met)
- Improved handling of budget, diet, and privacy requirements simultaneously

### ✅ **Negation Understanding**
- Specialized handling of "NOT", "never", "don't" phrases
- Better at finding exclusions and restrictions
- Improved performance on policy-based queries

### ✅ **Zero-Shot Generalization**
- Works on novel query types not seen during development
- Instruction templates provide flexible reasoning framework
- Maintains performance on unexpected query patterns

## Limitations and Next Steps

While Promptriever shows significant improvements, it still has limitations:
- Relies on embedding similarity (no explicit reasoning chains)
- Limited explainability of decisions
- No test-time compute for complex reasoning

**Next**: In Notebook 3, we'll explore **Rank1** - which adds explicit reasoning chains and test-time compute for even more sophisticated retrieval!